# Olmo 3 circuit-analysis smoke test

This notebook drives `circuit_analysis.py` (same folder) over a small bank of *"are you conscious?"*-style prompts and their controls, using [TransformerLens](https://github.com/TransformerLensOrg/TransformerLens).

The model is the project's workhorse, the Olmo 3 7B Instruct stack (`olmo_7b_instruct_stack` in `llm_consciousness_self_attribution/config/model_stacks.yaml`). The default stage is `instruct` (`allenai/Olmo-3-7B-Instruct`); set `STAGE` below to `'sft'` or `'dpo'` for the earlier checkpoints. Prompts use the model's own tokenizer and chat template, with no system message, as in the evals' baseline condition.

The notebook covers:

1. **Model outputs.** The greedy response and next-token distribution for each prompt.
2. **Residual stream.** The logit lens across layers, direct logit attribution per component, and residual norms.
3. **Attention heads.** Direct logit attribution per head, plus attention-pattern statistics (entropy, attention to the first token and to the self-referential tokens, previous-token score).
4. **Attention patterns.** Heatmaps for individual heads on individual prompts.
5. **Activation patching.** Which heads carry the difference between *"Are you conscious?"* and *"Are you hungry?"*.
6. **CSV export.** Model outputs and run metadata are saved to `outputs/`.

The measured scalar throughout is the **logit difference** between `'Yes'` and `'No'` as the first token of the assistant's turn.

> **Hardware.** A 7B model needs about 30 GB of memory in float32 (about 15 GB with `DTYPE = 'bfloat16'`). Use a GPU, e.g. the A100s the project uses on Modal; on CPU it is very slow. Activation patching is the slow part: 1,024 forward passes per prompt pair.

**Setup:** from the repo root run `uv sync --group interp`, then select the project's `.venv` as the kernel.

In [ ]:
import json
import sys
from dataclasses import asdict
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch

# Make the sibling module importable whether the kernel starts in this folder or at the repo root.
HERE = Path.cwd()
if not (HERE / 'circuit_analysis.py').exists():
    HERE = HERE / 'prototyping_scripts' / 'circuit_analysis'
sys.path.insert(0, str(HERE))

import circuit_analysis as ca

pd.set_option('display.max_colwidth', 80)

In [ ]:
# Run configuration. Defaults come from config/analysis.yaml (prompts from config/prompts.yaml);
# edit those files for lasting changes, or override a value here for this session only.
STAGE = ca.DEFAULT_STAGE                    # e.g. 'instruct', 'sft', 'dpo'
MODEL_NAME = ca.resolve_stage(STAGE).model  # from config/model_stacks.yaml
DTYPE = ca.DEFAULT_DTYPE                    # 'float32' or 'bfloat16'
SEED = ca.DEFAULT_SEED
SETTINGS = ca.DEFAULT_GENERATION            # e.g. dataclasses.replace(ca.DEFAULT_GENERATION, max_new_tokens=128)
OUTPUT_DIR = ca.DEFAULT_OUTPUT_DIR
RUN_ID = ca.new_run_id()

print(f'config: {ca.SETTINGS_FILE.name} + {ca.PROMPTS_FILE.name} (sha256 {ca.config_sha256()[:12]})')
print(f'run {RUN_ID}: {ca.STACK}/{STAGE} = {MODEL_NAME} dtype={DTYPE} seed={SEED} {SETTINGS} -> {OUTPUT_DIR}')

## 1. Load the tokenizer (Hugging Face `transformers`) and the model (TransformerLens)

The model's own tokenizer (with its chat template) comes from `transformers.AutoTokenizer` and is handed to TransformerLens, so both sides use the same tokenizer object.

`load_model` uses TransformerLens 4's `TransformerBridge.boot_transformers(MODEL_NAME, tokenizer=...)`, which picks the Olmo 3 adapter from the checkpoint's config, then `enable_compatibility_mode()` for the legacy hook names (`hook_pattern`, `hook_resid_post`, ...).

Olmo 3 applies RMSNorm *after* each sublayer, so TransformerLens can't fold its norms and warns that it skipped `fold_ln`. That warning is expected. The final norm's gain is therefore not folded into `W_U`, and the module multiplies it back into the direction used for direct logit attribution.

In [ ]:
ca.set_seed(SEED)
tokenizer = ca.load_tokenizer(MODEL_NAME)
print(type(tokenizer).__name__, '|', tokenizer.name_or_path, '| vocab size', len(tokenizer))

model = ca.load_model(tokenizer, MODEL_NAME, dtype=DTYPE)
cfg = model.cfg
print(f'{cfg.model_name}: {cfg.n_layers} layers x {cfg.n_heads} heads, '
      f'd_model={cfg.d_model}, device={cfg.device}, dtype={cfg.dtype}')
print('stop tokens:', {i: tokenizer.decode([i]) for i in ca.stop_token_ids(model)})

## 2. The prompt bank

The prompts and patching pairs live in `config/prompts.yaml`. The bank has five categories, so that a *consciousness* effect can be told apart from a *question-about-you* effect and from a plain *yes/no-question* effect:

- `self_consciousness`: consciousness attributed to the model (the condition of interest).
- `other_consciousness`: the same concept, attributed to something else.
- `self_control` / `other_control`: self- or other-referential questions that aren't about consciousness.
- `factual_control`: plain factual yes/no questions.

In [ ]:
pd.DataFrame([asdict(p) for p in ca.PROMPTS])

In [ ]:
# Tokenization checks: both answers must be single tokens, and the patching pairs must line up.
answer_ids = ca.answer_token_ids(model)
print('answer ids:', dict(zip([ca.POSITIVE_ANSWER, ca.NEGATIVE_ANSWER], answer_ids)))

# The full prompt as the model sees it (chat template, default system prompt, assistant opener).
example = ca.encode_prompt(model, ca.PROMPTS[0].question)
print(example.text)
print('question tokens:', [model.to_str_tokens(example.tokens[0])[i] for i in example.question_positions])

for clean_id, corrupt_id in ca.PATCHING_PAIRS:
    aligned = ca.tokens_aligned(model, ca.prompt_by_id(clean_id), ca.prompt_by_id(corrupt_id))
    print(f'patching pair {clean_id} -> {corrupt_id}: aligned={aligned}')

## 3. Run the analysis

This makes one generation plus one cached forward pass per prompt. `run_analysis` validates its own output: the row counts, no NaN logit differences, and that the direct logit attributions sum back to the logit difference.

In [ ]:
prompt_df, head_df = ca.run_analysis(model, ca.PROMPTS, SETTINGS, SEED, RUN_ID)

prompt_df[[
    'prompt_id', 'category', 'generated_first_line', 'top_next_token',
    'logit_diff', 'prob_positive', 'prob_negative', 'dla_reconstruction_error',
]]

In [ ]:
# Full model outputs.
for _, row in prompt_df.iterrows():
    print(f"--- {row['prompt_id']} ({row['category']})")
    print(row['formatted_prompt'] + row['generated_text'])

## 4. Yes-minus-No logit difference by prompt

In [ ]:
categories = list(dict.fromkeys(prompt_df['category']))
colors = dict(zip(categories, plt.cm.tab10.colors))

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(prompt_df['prompt_id'], prompt_df['logit_diff'], color=[colors[c] for c in prompt_df['category']])
ax.axhline(0, color='black', linewidth=0.8)
ax.set_ylabel('logit(Yes) - logit(No)')
ax.tick_params(axis='x', rotation=75)
ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=colors[c]) for c in categories], labels=categories, fontsize=8)
plt.tight_layout()
plt.show()

prompt_df.groupby('category')['logit_diff'].agg(['mean', 'std', 'count'])

## 5. Residual stream

### 5a. Logit lens

The accumulated residual stream at the final position after each sub-layer is passed through the final RMSNorm and projected onto the Yes-minus-No direction. This shows *where* in depth the answer preference forms. The last point equals the model's logit difference (Olmo 3 has no unembedding bias).

In [ ]:
def json_frame(column):
    """Expand a JSON-dict column into a DataFrame with one row per prompt."""
    return pd.DataFrame([json.loads(s) for s in prompt_df[column]], index=prompt_df['prompt_id'])

lens = json_frame('logit_lens')
lens_by_category = lens.groupby(prompt_df['category'].values).mean()

fig, ax = plt.subplots(figsize=(11, 4))
for category, values in lens_by_category.iterrows():
    ax.plot(range(len(values)), values.values, marker='.', label=category, color=colors[category])
ax.set_xticks(range(len(lens.columns)), lens.columns, rotation=75, fontsize=7)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_ylabel('logit diff (logit lens)')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

### 5b. Direct logit attribution per component

This is the contribution of each embedding, attention layer and MLP *output*, taken directly through the unembedding (indirect effects through later layers aren't counted).

In [ ]:
component_dla = json_frame('component_dla')
dla_by_category = component_dla.groupby(prompt_df['category'].values).mean()

fig, ax = plt.subplots(figsize=(11, 3.5))
limit = dla_by_category.abs().max().max()
im = ax.imshow(dla_by_category.values, cmap='RdBu', vmin=-limit, vmax=limit, aspect='auto')
ax.set_xticks(range(len(dla_by_category.columns)), dla_by_category.columns, rotation=75, fontsize=7)
ax.set_yticks(range(len(dla_by_category.index)), dla_by_category.index)
fig.colorbar(im, ax=ax, label='logit diff contribution')
plt.tight_layout()
plt.show()

In [ ]:
# Residual-stream norm at the final position, by depth.
norms = json_frame('resid_norms')
fig, ax = plt.subplots(figsize=(8, 3.5))
for category, values in norms.groupby(prompt_df['category'].values).mean().iterrows():
    ax.plot(range(len(values)), values.values, marker='.', label=category, color=colors[category])
ax.set_xticks(range(len(norms.columns)), norms.columns, rotation=75, fontsize=7)
ax.set_ylabel('L2 norm')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 6. Attention heads

### 6a. Direct logit attribution per head

The left panel is the mean over the `self_consciousness` prompts, the middle panel is the mean over `self_control`, and the right panel is their difference. The heads that light up only on the right are candidates for consciousness-specific (rather than self-reference-specific) processing.

In [ ]:
def head_grid(df, value):
    """[layer x head] grid of the mean of `value` over the rows of `df`."""
    return df.pivot_table(index='layer', columns='head', values=value, aggfunc='mean')

def plot_grids(grids, titles, cmap='RdBu', symmetric=True, label=''):
    fig, axes = plt.subplots(1, len(grids), figsize=(4.5 * len(grids), 4))
    axes = [axes] if len(grids) == 1 else axes
    for ax, grid, title in zip(axes, grids, titles):
        limit = grid.abs().max().max()
        kwargs = {'vmin': -limit, 'vmax': limit} if symmetric else {}
        im = ax.imshow(grid.values, cmap=cmap, **kwargs)
        ax.set_title(title, fontsize=10)
        ax.set_xlabel('head')
        ax.set_ylabel('layer')
        fig.colorbar(im, ax=ax, label=label, shrink=0.8)
    plt.tight_layout()
    plt.show()

conscious = head_grid(head_df[head_df['category'] == 'self_consciousness'], 'dla')
control = head_grid(head_df[head_df['category'] == 'self_control'], 'dla')
plot_grids([conscious, control, conscious - control],
           ['self_consciousness', 'self_control', 'difference'], label='head DLA')

In [ ]:
# Heads with the largest mean |DLA| on the consciousness prompts.
(head_df[head_df['category'] == 'self_consciousness']
 .groupby('head_label')['dla'].mean()
 .sort_values(key=abs, ascending=False)
 .head(10))

### 6b. Attention-pattern statistics (all prompts)

- **Entropy** (final position): low means the head attends sharply.
- **First-token attention**: attention to position 0 (`<|im_start|>`). A high value usually means the head is "off" for this input.
- **Self-reference attention**: the attention mass from the final position onto *you / your* tokens in the question. The default system prompt's "You are …" is not counted. It is NaN for prompts with no such token.
- **Previous-token score**: the mean attention from each position to the one before it, which identifies previous-token heads.

In [ ]:
stats = ['final_pos_entropy', 'final_pos_first_token_attention', 'final_pos_self_ref_attention', 'prev_token_score']
plot_grids([head_grid(head_df, s) for s in stats], stats, cmap='viridis', symmetric=False)

## 7. Attention patterns for one prompt

This shows the full query-by-key pattern for the head with the largest |DLA| on the chosen prompt. Change `PROMPT_ID`, or set `LAYER` / `HEAD` by hand.

In [ ]:
PROMPT_ID = 'self_conscious'

prompt = ca.prompt_by_id(PROMPT_ID)
tokens = ca.encode_prompt(model, prompt.question).tokens
str_tokens = model.to_str_tokens(tokens[0])
with torch.no_grad():
    _, cache = model.run_with_cache(tokens)

rows = head_df[head_df['prompt_id'] == PROMPT_ID]
top = rows.loc[rows['dla'].abs().idxmax()]
LAYER, HEAD = int(top['layer']), int(top['head'])
print(f"top-|DLA| head for {PROMPT_ID}: {top['head_label']} (DLA {top['dla']:.3f})")

pattern = cache['pattern', LAYER][0, HEAD].float().cpu()
labels = [repr(t) for t in str_tokens]
size = 0.22 * len(labels) + 2  # the chat template makes prompts ~40+ tokens long
fig, ax = plt.subplots(figsize=(size, size - 1))
im = ax.imshow(pattern, cmap='Blues', vmin=0, vmax=1)
ax.set_xticks(range(len(labels)), labels, rotation=90, fontsize=7)
ax.set_yticks(range(len(labels)), labels, fontsize=7)
ax.set_xlabel('key')
ax.set_ylabel('query')
ax.set_title(f'L{LAYER}H{HEAD} on {PROMPT_ID}')
fig.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

In [ ]:
# Where every head in LAYER looks from the final position.
final_rows = cache['pattern', LAYER][0, :, -1, :].float().cpu()
fig, ax = plt.subplots(figsize=(size, 6))
im = ax.imshow(final_rows, cmap='Blues', vmin=0, vmax=1, aspect='auto')
ax.set_xticks(range(len(labels)), labels, rotation=90, fontsize=7)
ax.set_ylabel('head')
ax.set_title(f'layer {LAYER}: attention from the final position')
fig.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

In [ ]:
# Optional interactive view. circuitsvis is not in the `interp` group: `uv pip install circuitsvis` to enable.
try:
    import circuitsvis as cv
except ImportError:
    print('circuitsvis not installed; skipping the interactive view')
else:
    display(cv.attention.attention_patterns(tokens=str_tokens, attention=cache['pattern', LAYER][0]))

## 8. Activation patching of head outputs

For each pair (clean -> corrupted, e.g. *conscious* -> *hungry*), one head at a time has its output (at all positions) from the clean run patched into the corrupted run.

- `effect` is the change in logit difference, in logits.
- `normalized_effect` scales that so 1.0 means the head alone restores the clean behaviour. It is NaN when the clean and corrupted logit differences are almost equal, because the ratio would be meaningless.

Pairs that don't tokenize to the same length are skipped with a warning (see the check in section 2).

In [ ]:
patch_df = ca.run_patching(model, ca.PATCHING_PAIRS, RUN_ID)

if patch_df.empty:
    print('no aligned pairs; nothing patched')
else:
    pairs = patch_df.groupby(['clean_prompt_id', 'corrupt_prompt_id'])
    print(pairs[['clean_logit_diff', 'corrupt_logit_diff']].first())
    grids = [head_grid(df, 'effect') for _, df in pairs]
    titles = [f'{clean} -> {corrupt}' for (clean, corrupt), _ in pairs]
    plot_grids(grids, titles, label='patching effect (logits)')

## 9. Save to CSV

| File | One row per | Contents |
| --- | --- | --- |
| `<run_id>_prompts.csv` | prompt | the question, the full chat-formatted prompt, tokens, generated response, top next tokens, Yes/No logits and probabilities, logit lens, component DLA, residual norms, top heads, and the run metadata (stack, stage, model, tokenizer, device, dtype, system prompt, seed, generation settings, library versions) |
| `<run_id>_heads.csv` | prompt x layer x head | head DLA and attention-pattern statistics |
| `<run_id>_patching.csv` | pair x layer x head | patched logit diff and effect |

Nested values (token lists, per-layer dicts) are stored as JSON strings, and missing values are written as `NaN`.

In [ ]:
paths = ca.save_results(prompt_df, head_df, patch_df, OUTPUT_DIR, RUN_ID)
for kind, path in paths.items():
    print(f'{kind}: {path}')

# Round-trip check: read the prompts CSV back and make sure it is what was written.
reloaded = pd.read_csv(paths['prompts'])
assert len(reloaded) == len(prompt_df)
assert json.loads(reloaded.loc[0, 'prompt_str_tokens']) == json.loads(prompt_df.loc[0, 'prompt_str_tokens'])
reloaded.head()

## 10. Reading the results, and next steps

- **n is tiny.** There are a handful of prompts per category and one greedy sample each, so treat any category difference as a hint at most.
- **DLA is direct effect only.** A head with a small DLA can still matter through later layers. Patching is the causal check.
- **The Yes/No frame is a choice.** The model may prefer to open its response with something other than `'Yes'` / `'No'` (check `top_next_token`). On the earlier GPT-2 Instruct run, the real competitor on self-questions was `' I'` (as in "I do not have feelings"). If so here, the logit difference measures a low-probability choice. Compare against `factual_control` before reading anything into `self_consciousness`. The answer tokens are set in `config/analysis.yaml`.
- **The system prompt is part of the input.** With no system message the chat template adds Olmo 3's default ("You are a helpful function-calling AI assistant…"). Set `prompt.system_prompt` in `config/analysis.yaml` to override it, e.g. to mirror the evals' identity conditions.
- **Across the training stack:** run this notebook with `STAGE = 'sft'`, `'dpo'` and `'instruct'` and compare the logit-diff bars, logit lens and head grids. That's the comparison the project's behavioural evals make.